In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
import pprint

llm = ChatOpenAI(
    model="gpt-5.6-luna",
    use_responses_api=True,
)

response = llm.invoke([HumanMessage("잘 지냈어?")])
pprint.pprint(response.model_dump(), width=500)

{'additional_kwargs': {},
 'content': [{'annotations': [], 'id': 'msg_0266813e2b5016b2006aa2eb1f919087d0bb435df90c4079b6', 'phase': 'final_answer', 'text': '응, 잘 지냈어! 😊 너는 어때?', 'type': 'text'}],
 'id': 'resp_0266813e2b5016b2006aa2eb1f0d5887d08d5a37dbcf4390e3',
 'invalid_tool_calls': [],
 'name': None,
 'response_metadata': {'created_at': 1789061919.0, 'id': 'resp_0266813e2b5016b2006aa2eb1f0d5887d08d5a37dbcf4390e3', 'metadata': {}, 'model': 'gpt-5.6-luna', 'model_name': 'gpt-5.6-luna', 'model_provider': 'openai', 'object': 'response', 'service_tier': 'default', 'status': 'completed'},
 'tool_calls': [],
 'type': 'ai',
 'usage_metadata': {'input_token_details': {'cache_creation': 0, 'cache_read': 0}, 'input_tokens': 11, 'output_token_details': {'reasoning': 0}, 'output_tokens': 17, 'total_tokens': 28}}


### ```@tool```: LangChain에서 일반 Python 함수를 LLM이 사용할 수 있는 “도구(Tool)”로 등록하는 데 쓰는 데코레이터
- @tool을 붙이면 get_current_time 함수가 LangChain Tool 객체로 변환되기 때문에 tools나 tool_dict에 넣어서 사용할 수 있음

### 이전 방식
```txt
Python 함수
   ↓
tools = [{"type": "function", ...}]  ← 우리가 직접 스키마 작성
   ↓
OpenAI API
```
--------------
### LangChain 방식
```txt
@tool
def get_current_time(...):
   ↓
tools = [get_current_time]           ← 함수만 넣음
   ↓
llm.bind_tools(tools)                ← LangChain이 필요한 tool 정보로 변환
```

(복습)
### tools = [get_current_time] → Tool들을 순서대로 모아놓은 리스트라서, 여러 Tool을 LLM에 전달할 때 주로 사용함,
### tool_dict = {"get_current_time": get_current_time} → Tool 이름을 key로 해서 바로 찾을 수 있게 만든 딕셔너리라서, LLM이 요청한 함수 이름으로 실제 함수를 찾아 실행할 때 사용함. 

In [5]:
import sys
sys.path.append("..")

from langchain_tool_functions  import tools, tool_dict
# 도구를 모델에 바인딩: 모델에 도구를 바인딩하면, 도구를 사용하여 llm 답변을 생성할 수 있음
llm_with_tools = llm.bind_tools(tools)

- Tool이 실제로 등록되어 있다면 LLM은 질문을 보고 알아서 적절한 Tool을 선택할 수 있음
- SystemMessage의 “Tool을 사용할 수 있다”는 문장은 필수라기보다 LLM에게 Tool 사용을 명시적으로 안내하는 추가 지침에 가까움

In [6]:
from langchain_core.messages import SystemMessage

# (4) 사용자의 질문과 tools 사용하여 llm 답변 생성
messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("부산은 지금 몇시야?"),
]

# (5) llm_with_tools를 사용하여 사용자의 질문에 대한 llm 답변 생성
response = llm_with_tools.invoke(messages)
messages.append(response)

# (6) 생성된 llm 답변 출력
print(messages)

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}), AIMessage(content=[{'id': 'rs_092665286a0e8217006aa2eb21235887d08b80409a9f2d9b35', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABqoush0sRHuUAhwhwOJUWw-PXA38vHHXIDdlc8e4_Wai5XamXAIFiQhcA6WReXyo1xSFJ7nGLZibaN9RKq-LfIyuuQ1AwJWUKCAA7Tu1bEe2e5P4Ls_hc86SvlqknPsrw8L1LLhfeCHhiyOYYB7dP0SfDivTIoNfTqjfMyHWxGz8_Ig9lauHfYuTrPrHdzW8BVlx_LBFxiokjkTLduhW096RLZUpRLvv6ZhWeA0h495yYItowOLGOo0cgKWBn5bi-1bN6MFN8TRu0mGy2zaX_jw2X1IFkOevpw7W3oC3tSGtVmBQjfx6ngizVFQ_E3chXHPjteBUk985NlK-zh9-ds3uPWu9VerdZcR8wuHTue-9PIdfUI1EpTZYshjl3VaR_a3xtOchT1l2l0qrFzRYzwzYj5c8kh7INCePCZOA3exMcgD5ElrqxIySIpn-tnbueC8MES22i-K-Vwt-sj3oayeIlBhOflBUsi7jckIJ1ZYFnQ78Fz9WsJ7U2XFgybdRk0z0U83odg9TB2FCcdTnquQs80jyjCr6Vi7tt-suiP_9uz31Yxtpx8V4Byhus01BSNi5c6C6Cds9W1YJzbGxHGXpwXrEMvD43RE-4uiIj7XYUUHyTdo5M72--iUfGe1OvwRgHWU

- LLM이 요청한 Tool을 실제로 실행하고 그 결과를 messages에 추가하는 단계

In [7]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]] # (7) tool_dict를 사용하여 도구 함수를 선택
    print(tool_call["args"]) # (8) 도구 호출 시 전달된 인자 출력
    tool_msg = selected_tool.invoke(tool_call) # (9) 도구 함수를 호출하여 결과를 반환
    messages.append(tool_msg)

data = response.model_dump()
for item in data["content"]:
    item.pop("encrypted_content", None)

pprint.pprint(data, width=250)

{'timezone': 'Asia/Seoul', 'location': '부산'}
Asia/Seoul (부산) 현재시각 2026-09-11 02:38:42 
{'additional_kwargs': {},
 'content': [{'content': [], 'id': 'rs_092665286a0e8217006aa2eb21235887d08b80409a9f2d9b35', 'summary': [], 'type': 'reasoning'},
             {'arguments': '{"timezone":"Asia/Seoul","location":"부산"}',
              'call_id': 'call_MiaqlQZQakaOEH4GyrtoDWLp',
              'id': 'fc_092665286a0e8217006aa2eb21600087d09a8d2e5cefdc21c4',
              'name': 'get_current_time',
              'status': 'completed',
              'type': 'function_call'}],
 'id': 'resp_092665286a0e8217006aa2eb209ae087d0a8d93a7ea7c1c768',
 'invalid_tool_calls': [],
 'name': None,
 'response_metadata': {'created_at': 1789061920.0,
                       'id': 'resp_092665286a0e8217006aa2eb209ae087d0a8d93a7ea7c1c768',
                       'metadata': {},
                       'model': 'gpt-5.6-luna',
                       'model_name': 'gpt-5.6-luna',
                       'model_provider': 'op

- tool_calls → get_current_time 호출 결정 + timezone="Asia/Seoul", location="부산" 전달
- content의 function_call → Responses API가 실제로 요청한 함수 호출 정보 (call_id, arguments, name)
---------------

- Tool 실행 결과까지 포함된 messages를 다시 LLM에게 보내서, 최종 자연어 답변을 받는 단계

In [8]:
llm_with_tools.invoke(messages)

AIMessage(content=[{'type': 'text', 'text': '부산은 현재 **2026년 9월 11일 오전 2시 38분**입니다.', 'annotations': [], 'id': 'msg_092665286a0e8217006aa2eb230cd487d0a08d966537951acc', 'phase': 'final_answer'}], additional_kwargs={}, response_metadata={'id': 'resp_092665286a0e8217006aa2eb224a7087d0b4699ad399deac5c', 'created_at': 1789061922.0, 'metadata': {}, 'model': 'gpt-5.6-luna', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna'}, id='resp_092665286a0e8217006aa2eb224a7087d0b4699ad399deac5c', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 286, 'output_tokens': 28, 'total_tokens': 314, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}, 'output_token_details': {'reasoning': 0}})